In [1]:
import surprise
import pandas as pd
import numpy as np


import os
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\ml-100k') 

In [3]:
ratings = pd.read_csv('u.data' , sep='\t', names= ['uid','iid','rating','ts'])
ratings.drop('ts', axis = 1, inplace = True)
ratings.head()   

,uid,iid,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1


In [4]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()
lowest_rating,highest_rating

(np.int64(1), np.int64(5))

In [5]:
reader =surprise.Reader(rating_scale=(lowest_rating, highest_rating))

data = surprise.Dataset.load_from_df(ratings,reader)
type(data)

surprise.dataset.DatasetAutoFolds

In [6]:
similarity_options = {'name':'cosine', 'user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())
# calculate the expected ratings for all the users

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold 

In [8]:
param_grid = {
    'n_epochs': [50],
    'n_factors':[100,50],
    'lr_all': np.linspace(0.001,1,5),
    'reg_all': np.linspace(0.01, 0.8, 5)
}
kfold = KFold(n_splits = 5, random_state = 26, shuffle = True)

gs = GridSearchCV(surprise.SVD , param_grid , measures = ['rmse','mae'], joblib_verbose = 3, cv = kfold , n_jobs = -1)
gs.fit(data)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:   35.9s
[Parallel(n_jobs=-1)]: Done 250 out of 250 | elapsed:  1.2min finished


In [9]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.9464266446018815
{'n_epochs': 50, 'n_factors': 50, 'lr_all': np.float64(0.001), 'reg_all': np.float64(0.01)}


In [10]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

In [17]:
iids = ratings['iid'].unique()
print(len(iids))

1682


In [18]:
user = 50
u_iid = ratings[ratings['uid']==user]['iid'].unique()
print('List of items rated by user: ',u_iid)
print("\nNo of items rated by user {0}: {1} ".format(user, len(u_iid)))

iids_to_predict = np.setdiff1d(iids, u_iid)

print("\nItems not rated by the user or those items for which the expected ratings are to be predict {0}".format(iids_to_predict))

List of items rated by user:  [ 246  823  253  475 1084  286    9  125  123  325  508  288  319  324
  276 1008 1010  268  544   15  327  124  547  100]

No of items rated by user 50: 24 

Items not rated by the user or those items for which the expected ratings are to be predict [   1    2    3 ... 1680 1681 1682]


In [19]:
testset = [[user, iid, 0.] for iid in iids_to_predict]
predictions = algo.test(testset)

exp_ratings = [(predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
exp_ratings = pd.DataFrame(exp_ratings, columns=['iid', 'est_rating'])

# Sort the DataFrame instead of the list
exp_ratings.sort_values(by='est_rating', ascending=False).head(30)

,iid,est_rating
305,318,4.743770
391,408,4.646956
61,64,4.606045
10,12,4.530985
410,427,4.479921
162,169,4.476547
465,483,4.471156
462,480,4.445641
167,174,4.427628
47,50,4.427266


In [20]:
os.chdir(r'E:\PGCP-AI\PML\Cases\SURPRISE\ml-100k') 

In [21]:
movies = pd.read_csv('movies_list.csv', encoding = 'latin-1')
movies.columns = movies.columns.str.strip()
movies.columns

Index(['movie id', 'movie title', 'release date', 'video release date',
       'IMDb URL', 'unknown', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [22]:
exp_ratings = exp_ratings.merge(
    movies[['movie id', 'movie title', 'release date']],
    how='left',
    left_on='iid',
    right_on='movie id'
)
exp_ratings.sort_values(by='est_rating', ascending=False).head(10)

,iid,est_rating,movie id,movie title,release date
305,318,4.743770,318,Schindler's List (1993),01-Jan-93
391,408,4.646956,408,"Close Shave, A (1995)",28-Apr-96
61,64,4.606045,64,"Shawshank Redemption, The (1994)",01-Jan-94
10,12,4.530985,12,"Usual Suspects, The (1995)",14-Aug-95
410,427,4.479921,427,To Kill a Mockingbird (1962),01-Jan-62
162,169,4.476547,169,"Wrong Trousers, The (1993)",01-Jan-93
465,483,4.471156,483,Casablanca (1942),01-Jan-42
462,480,4.445641,480,North by Northwest (1959),01-Jan-59
167,174,4.427628,174,Raiders of the Lost Ark (1981),01-Jan-81
47,50,4.427266,50,Star Wars (1977),01-Jan-77
